A set of simulations & hypothesis tests for assessing the statistical power of "minP vs CRE of interest" tests in the cohen dataset...

I have not included a lot of explanations because this notebook is extremely similar to the two_thirds_inactive for shendure, which has lots of explanations. Take a look at those if you are having trouble undersanting this notebook...

# Imports & dask cluster creation

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
from pathlib import Path

%load_ext autoreload
%autoreload 2

In [ ]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client=Client(cluster)
else:
    cluster=SLURMCluster(
        cores=3,#cores per slurm job
        memory="512G",#memory per slurm job
        processes=3,#dask workers per slurm job,
        job_extra_directives=["-p week", 
            f"--job-name=simclust_worker",
            "--resources \"FIT=1\"",
            f"--time=72:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=2)
    #cluster.scale(jobs=20)
    #cluster.scale(jobs=20)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

In [ ]:
data_root=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
client.dashboard_link

# Ground truth creation

In [ ]:
primordial=scm.ortho.load(client,path=data_root,name="cohen_ortho")
primordial.compute_model_qc()

In [ ]:
import pandas as pd
import numpy as np

vals=[]
for key in primordial.by_cell_qc.keys():
    working=primordial.by_cell_qc[key]['dat'].reset_index().drop(columns=["mean(umis_mpra_bc)"])
    working["cell_type"]=key
    vals.append(working)
gt_cell_type=pd.concat(vals)

In [ ]:
#casting away from sparse, since it's not sparse anymore
gt_cell_type["mu"] = gt_cell_type["mu"].astype(float)


In [ ]:
gt_cell_type

In [ ]:
combo_counts=gt_cell_type[gt_cell_type["cre_id"]!="reference"].drop(columns=["mu"]).groupby("cell_type").nunique()
max_tfection=max(combo_counts["cre_id"])
combo_counts

So quite unlike shendure we have total transfection..!

In [ ]:
gt_cell_type=gt_cell_type[gt_cell_type["cre_id"]!="reference"]

In [ ]:
real_means=gt_cell_type.drop(columns=["cre_id","cell_type"])
real_means

In [ ]:
num_cell_types=len(gt_cell_type["cell_type"].unique())
num_cell_types

In [ ]:
synth_cre_names=[f"synthcre_{i}" for i in range(0,len(real_means))]

parts=[]
for i in range(0,num_cell_types):
    working=real_means.sample(frac=1)
    working["cre_id"]=synth_cre_names
    working["cell_type"]=f"ct_{i}"
    parts.append(working)

cartesian=pd.concat(parts).sample(frac=1).reset_index(drop=True)
cartesian

In [ ]:
#make 70% inactive
minP=scm.COHEN_BOUNDS.reference_activity
num_inactive=int(len(cartesian)*0.7)
cartesian.loc[cartesian.index[:num_inactive],"mu"]=minP
cartesian

In [ ]:
#now we pick max_tfection CREs to proceed with, because this is the number of unique cre_id 
#found in the the cell type with the most unique CRE ids (all are the same in cohen)

chosen_cre_id=np.random.choice(cartesian["cre_id"].unique(),max_tfection,replace=False)
sampled_nbs=cartesian[cartesian["cre_id"].isin(chosen_cre_id)]
sampled_nbs

In [ ]:
ref_df=pd.DataFrame({"cell_type":[f"ct_{i}" for i in range(0,num_cell_types)]})
ref_df["mu"]=minP
ref_df["cre_id"]="reference"
ref_df


In [ ]:
final_gt=pd.concat([ref_df,sampled_nbs])
final_gt["cell_type"]=final_gt["cell_type"].replace({"ct_0":"reference"})
final_gt

# Creating artificial libraries

In [ ]:
libraries=[scm.simulate_library(CREs=final_gt["cre_id"],
                 library_model=scm.COHEN_BOUNDS.library_model)
                 for i in range(5)]

# Creating sim


In [ ]:
bound=scm.COHEN_BOUNDS.copy()

In [ ]:
print(bound.cells_per_cell_type.name)
print(bound.cells_per_cell_type.index.name)
s=bound.cells_per_cell_type
s

In [ ]:
s.index=[f"ct_{i}" for i in range(1,num_cell_types)]+["reference"]
s

In [ ]:
bound.cells_per_cell_type=s

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-02-05",
                            client=client,
                            libraries=libraries,
                            library_mapping="corresponding",
                            flatten_overtransfection=True,
                            n_sims=5,
                            experiment_bounds=bound,
                            ground_truth=final_gt)

In [ ]:
sim.gamut()

In [ ]:
sim.save()

# Fit orthos

In [ ]:
sim=scm.de_novo_simulation(location=data_root,
                            name="twothird_pow_sim_2026-02-05",
                            client=client)

In [ ]:
sim.fit_orthos(serial_orthos=False,direction="by_cell_type")

In [ ]:
sim.save()

# Shutdown

In [ ]:
client.close()
cluster.close()